# RL-дообучение последовательной рекомендательной модели (GPT-2 + Reward Model + RL)

Ноутбук воспроизводит полный эксперимент на GPU (T4/A100):
1. **KION** (МТС): base GPT-2 next-item → reward model (watched_pct) → RL (REINFORCE/GRPO, варианты награды) → метрики Recall@10 / NDCG@10 / MAP@10, разнообразие, rewach@10, многошаговая генерация.
2. **Yambda** (Яндекс Музыка, subsample из sequential/50m listens.parquet): тот же пайплайн для проверки переносимости.

**Подготовка:** упакуйте локально папку проекта (`src/`) в `rlrec_kion_src.zip`
и загрузите её через меню *Файлы* этого блокнота (иконка 📁 слева), либо скормите
ноутбуку публичный GitHub-репозиторий в ячейке ниже.

In [ ]:
#@title 0. Зависимости и распаковка кода
%%shell
set -e
pip install -q numpy
pip install -q datasets pyarrow   # нужен только для Yambda
mkdir -p /content/rlrec && cd /content/rlrec
if [ -f /content/rlrec_kion_src.zip ]; then
  unzip -o -q /content/rlrec_kion_src.zip
fi
# Опционально: раскомментируйте и вставьте URL своего репозитория
# git clone --depth 1 https://github.com/<you>/<repo>.git /content/rlrec
ls -R /content/rlrec | head -50

In [ ]:
#@title 1. Данные: скачать KION
%%shell
set -e
cd /content/rlrec
mkdir -p data/kion && cd data/kion
[ -f interactions.csv ] || (curl -sL -o interactions.csv.zip https://raw.githubusercontent.com/irsafilo/KION_DATASET/main/interactions.csv.zip && unzip -o -q interactions.csv.zip)
wc -l interactions.csv

In [ ]:
#@title 2. Конфиг GPU-прогона
import sys, os, json
sys.path.insert(0, '/content/rlrec')
from src.config import Config

BASE = dict(dataset='kion', processed_dir='data/processed_kion', run_name='kion_completion',
            d_model=128, n_layer=4, n_head=8, d_ff=512, dropout=0.2, max_len=50,
            min_user_events=5, min_item_freq=5,
            base_epochs=8, base_batch=512, base_lr=3e-4,
            rm_epochs=8, rm_batch=512,
            rl_steps=4000, rl_batch=128, rl_lr=1e-5, kl_beta=0.05,
            eval_every=500, rl_eval_users=8000, gen_users=1000, topk=10)

def make_cfg(**over):
    c = Config(**{**BASE, **over})
    return c

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
#@title 3. Base-модель + Reward model + оценка RM (один раз, переиспользуется)
from src.data import UserData, prepare_kion
from src.model import RewardModel
from src import train as T
import src.main as MAIN
import torch

cfg = make_cfg()
os.makedirs(cfg.run_dir, exist_ok=True)
if not os.path.exists(cfg.processed_npz_path):
    prepare_kion(cfg)
data = UserData(cfg); cfg.vocab = data.vocab
print('users', data.n_users, 'events', len(data.seqs), 'vocab', data.vocab)

if not os.path.exists(cfg.policy_path):
    T.train_base(cfg, data, DEVICE)
if not os.path.exists(cfg.rm_path):
    T.train_rm(cfg, data, DEVICE)
rm = RewardModel(cfg).to(DEVICE); rm.load_state_dict(torch.load(cfg.rm_path, map_location=DEVICE))
rmeval = MAIN.eval_rm(cfg, data, rm, DEVICE)
print('[rmeval]', json.dumps(rmeval, indent=1))
json.dump(rmeval, open('runs/rmeval.json', 'w'), indent=1)

In [ ]:
#@title 4. RL-варианты: награда x алгоритм x KL
VARIANTS = [
  dict(run_name='kion_completion',        rl_algo='reinforce', rl_reward='completion',           kl_beta=0.05),
  dict(run_name='kion_completion_nokl',   rl_algo='reinforce', rl_reward='completion',           kl_beta=0.0),
  dict(run_name='kion_completion_norep',  rl_algo='reinforce', rl_reward='completion_norep',     kl_beta=0.05),
  dict(run_name='kion_completion_poppen', rl_algo='reinforce', rl_reward='completion_pop_pen',   kl_beta=0.05),
  dict(run_name='kion_grpo',              rl_algo='grpo',      rl_reward='completion',           kl_beta=0.05),
]
for v in VARIANTS:
    cfg = make_cfg(**v)
    os.makedirs(cfg.run_dir, exist_ok=True)
    print('='*20, cfg.run_name)
    T.rl_finetune(cfg, data, DEVICE)
    from src.model import PolicyModel
    pol = PolicyModel(cfg).to(DEVICE)
    pol.load_state_dict(torch.load(f'runs/{cfg.run_name}/policy_rl.pt', map_location=DEVICE))
    ev = T.evaluate_policy(cfg, data, pol, 'test', DEVICE, extra_rm_reward=True, rm=rm)
    gen = None
    print(cfg.run_name, json.dumps(ev))
    json.dump(ev, open(f'runs/{cfg.run_name}/eval_final.json', 'w'), indent=1)

In [ ]:
#@title 5. Многошаговая генерация base vs RL
from src.model import PolicyModel
from src import metrics as MM
base_pol = PolicyModel(make_cfg()).to(DEVICE)
base_pol.load_state_dict(torch.load(make_cfg().policy_path, map_location=DEVICE))
out = {}
for name, fn in [('base', make_cfg().policy_path)] + [(v['run_name'], f"runs/{v['run_name']}/policy_rl.pt") for v in VARIANTS]:
    p = PolicyModel(make_cfg()).to(DEVICE)
    p.load_state_dict(torch.load(fn, map_location=DEVICE))
    out[name] = MM.multi_step_generate(make_cfg(), p, data, rm, DEVICE)
print(json.dumps(out, indent=1))
json.dump(out, open('runs/multistep.json', 'w'), indent=1)

In [ ]:
#@title 6. Yambda (subsample) — переносимость метода
#@markdown Осторожно: streaming-проход по sequential/50m listens.parquet занимает ~20-40 мин.
from src.config import Config as C2
from src.data import prepare_yambda, UserData as UD2
ycfg = C2(**{**BASE, 'dataset': 'yambda', 'processed_dir': 'data/processed_yambda',
             'run_name': 'yambda_completion', 'min_item_freq': 20})
if not os.path.exists(ycfg.processed_npz_path):
    prepare_yambda(ycfg, n_users_target=30000, min_events=10, max_events=2_000_000)
ydata = UD2(ycfg); ycfg.vocab = ydata.vocab
T.train_base(ycfg, ydata, DEVICE); T.train_rm(ycfg, ydata, DEVICE)
T.rl_finetune(ycfg, ydata, DEVICE)
yp = PolicyModel(ycfg).to(DEVICE)
yp.load_state_dict(torch.load('runs/yambda_completion/policy_rl.pt', map_location=DEVICE))
yrm = RewardModel(ycfg).to(DEVICE); yrm.load_state_dict(torch.load(ycfg.rm_path, map_location=DEVICE))
print(json.dumps(T.evaluate_policy(ycfg, ydata, yp, 'test', DEVICE, extra_rm_reward=True, rm=yrm)))

In [ ]:
#@title 7. Скачать результаты
%%shell
cd /content/rlrec && zip -r /content/rlrec_results.zip runs data/processed_kion/*.json data/processed_yambda/*.json 2>/dev/null
ls -la /content/rlrec_results.zip